# 07 — End-to-end demo of the clause-review assistant (bonus T4)

Owner: Robin. One clause in, three answers out:

1. **Clause type**: the 3 most probable LEDGAR types, from the final Legal-BERT classifier
   (`legal-bert_final_20261019-1523`: test macro-F1 0.7905);
2. **Decision**: *automatic* if the model's confidence reaches the human-review threshold **0.892** (chosen on
   validation, A9: 78.9 % of the test clauses handled automatically at precision 0.949), otherwise *send to a lawyer*;
3. **Evidence**: the 5 most similar clauses of the training bank (dense retrieval index).

A lawyer can also ask a free-text question (retrieval only).

**Nothing is trained or tuned here.** The demo clauses are illustrations, not a new evaluation: the three test
clauses were picked by hand from the saved test predictions (one of each case), so no number in this notebook should
be reported as a score.

**Needs** the final checkpoint in `models/legal-bert_final/` (gitignored; the 4 files to download from Drive are
listed in the README, *Demo* section, and in the error message if they are missing). The first run builds and caches
the retrieval index over the 60,000 training clauses in `data/indexes/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import warnings

os.environ["TQDM_DISABLE"] = "1"  # no progress bars in a demo for the jury
warnings.filterwarnings("ignore")

import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import display
from transformers.utils import logging as transformers_logging

from src.config import RESULTS_DIR
from src.data import load_ledgar, to_pandas
from src.pipeline import ClauseReviewer
from src.utils import set_seed

set_seed(42)
pd.set_option("display.max_colwidth", None)
hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

reviewer = ClauseReviewer.from_local()

W1025 17:19:52.711000 29420 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Classifier: models/legal-bert_final on mps | review threshold 0.892
Retrieval index (BAAI/bge-small-en-v1.5, 60,000 training clauses) loaded from cache in 3.4 s


The retrieval encoder is the one ranked first on P@5 in `results/retrieval_summary.csv` (BGE-small). That
benchmark used a stratified 5 % sample of *test* queries, so this choice is a convenience for the demo, not a
validated model selection.

In [3]:
def show_review(title: str, text: str, gold: str | None = None) -> dict:
    """Review one clause and display the clause, its top-3 types, the decision and the closest training clauses."""
    result = reviewer.review(text)
    print(f"{title}\n{'=' * len(title)}\n{text}\n")
    top3 = pd.DataFrame(result["top3"])[["label_name", "probability"]].round(3)
    top3.index = range(1, len(top3) + 1)
    top3.columns = ["clause type", "probability"]
    if gold is not None:
        top3["gold label"] = ["✔" if name == gold else "" for name in top3["clause type"]]
    display(top3)
    verdict = "AUTOMATIC" if result["decision"] == "automatic" else "SEND TO A LAWYER"
    print(f"Decision: {verdict} (confidence {result['confidence']:.3f} vs threshold {result['threshold']:.3f})"
          + (f" | gold label: {gold}" if gold is not None else ""))
    similar = pd.DataFrame(result["similar"])[["score", "label_name", "snippet"]].round(3)
    similar.index = range(1, len(similar) + 1)
    similar.columns = ["cosine", "label", "first 200 characters"]
    display(similar)
    return {"clause": title.split(" — ")[0], "gold": gold or "—", "predicted": result["top3"][0]["label_name"],
            "confidence": round(result["confidence"], 3), "decision": result["decision"],
            "top retrieved label": result["similar"][0]["label_name"]}


summary = []

## 1. Three clauses of the 2019 test set

Read from LEDGAR by their position in the test split (`idx`). The case each one illustrates comes from the saved test
predictions of the final run; the demo re-runs the model locally (fp32, MPS/CPU instead of fp16 on a T4), so the
probabilities can differ from the saved ones in the last digits.

In [4]:
test_df = to_pandas(load_ledgar(), "test").set_index("idx")
errors_read = pd.read_csv(RESULTS_DIR / "error_analysis_reading.csv").set_index("idx")

TEST_CLAUSES = {
    9690: "Test clause 9690 — confident and right",
    9550: "Test clause 9550 — right, but below the threshold: sent to a lawyer",
    1201: "Test clause 1201 — confident error (from the 20 errors read in notebook 03)",
}
display(errors_read.loc[[1201], ["true_name", "pred_name", "tfidf_pred_name", "proba_pred", "tag", "note"]])

,true_name,pred_name,tfidf_pred_name,proba_pred,tag,note
idx,,,,,,
1201,Integration,Entire Agreements,Entire Agreements,0.9601,overlapping labels,one-sentence entire-agreement clause; Integration vs Entire Agreements are synonyms


In [5]:
idx = 9690
summary.append(show_review(TEST_CLAUSES[idx], test_df.loc[idx, "text"], test_df.loc[idx, "label_name"]))

Test clause 9690 — confident and right
Effective September 27, 2018, Executive shall receive an annual base salary of $1,161,840. Subsequent salary amounts shall be determined by the Company in its sole discretion; provided , however , that none of such subsequent annualized salaries shall be less than $1,161,840.



,clause type,probability,gold label
1,Base Salary,0.994,✔
2,Benefits,0.002,
3,Solvency,0.000,


Decision: AUTOMATIC (confidence 0.994 vs threshold 0.892) | gold label: Base Salary


,cosine,label,first 200 characters
1,0.940,Base Salary,"Executive shall receive a base salary (as applicable, the “ Base Salary ”) at an annual rate of no less than $1,050,000. Executive’s Base Salary shall be reviewed by the Company in 2017 and at least a"
2,0.936,Base Salary,"Executive shall receive an annual base salary of $435,000. Executive’s salary shall be paid semi-monthly and subject to all appropriate federal and state withholding taxes and shall be payable in acco"
3,0.934,Base Salary,"Effective as of July 1, 2017 Executive will be entitled to receive a base salary (“ Base Salary ”) of $300,000. Executive will receive a three percent (3%) annual salary increase in each subsequent y"
4,0.931,Base Salary,"During the Term, the Company shall pay the Executive a base salary (“ Base Salary ”) at the annual rate of (i) $512,500 for the period commencing on the Effective Date and ending June 30, 2019 and (ii"
5,0.929,Base Salary,"Effective retroactive to January 1, 2016, and during the Employment Period, the Company shall pay Executive an annual base salary in the amount of $450,000 (the “ Annual Base Salary ”), payable in ins"


In [6]:
idx = 9550
summary.append(show_review(TEST_CLAUSES[idx], test_df.loc[idx, "text"], test_df.loc[idx, "label_name"]))

Test clause 9550 — right, but below the threshold: sent to a lawyer
No waiver by the Corporation of any breach of this Agreement shall be a waiver of any preceding or succeeding breach. No waiver by the Corporation of any right under this Agreement shall be construed as a waiver of any other right. The Corporation shall not be required to give notice to enforce strict adherence to all terms of this Agreement.



,clause type,probability,gold label
1,Waivers,0.844,✔
2,No Waivers,0.141,
3,Miscellaneous,0.002,


Decision: SEND TO A LAWYER (confidence 0.844 vs threshold 0.892) | gold label: Waivers


,cosine,label,first 200 characters
1,0.907,Waivers,"No waiver by any of the Members of any breach or failure to comply with any provision of this Agreement shall be construed as, or constitute, a continuing waiver of such provision or a waiver of any o"
2,0.903,No Waivers,Any waiver by either party of the breach of any provision of this Agreement shall not operate or be construed as a waiver of any subsequent breach. The failure of either party at any time to enforce a
3,0.894,Waivers,The waiver by the Company of a breach or threatened breach of any obligation of this Agreement by the Employee shall not be construed as a waiver of any subsequent breach by the Employee. The refusal
4,0.890,Waivers,"The Corporation reserves the right to waive all or any part of the terms and conditions of the Covenants under appropriate circumstances, in its sole discretion. Any such waiver must be in writing and"
5,0.890,No Waivers,No waiver by any party of any breach of any terms or conditions contained herein shall be construed as a waiver of any other breach of the same or other terms or conditions herein. No delay or failure


In [7]:
idx = 1201
summary.append(show_review(TEST_CLAUSES[idx], test_df.loc[idx, "text"], test_df.loc[idx, "label_name"]))

Test clause 1201 — confident error (from the 20 errors read in notebook 03)
Except as stated herein, this is the parties’ entire agreement on the subject matter hereof, superseding all prior representations, negotiations or agreements.



,clause type,probability,gold label
1,Entire Agreements,0.960,
2,Integration,0.035,✔
3,Miscellaneous,0.001,


Decision: AUTOMATIC (confidence 0.960 vs threshold 0.892) | gold label: Integration


,cosine,label,first 200 characters
1,0.950,Integration,This Agreement constitutes the entire agreement among the Parties hereto pertaining to the subject matter hereof and supersedes all prior agreements and understandings pertaining thereto.
2,0.948,Entire Agreements,"This Agreement constitutes the entire agreement between the parties hereto with respect to the subject matter hereof, and supersedes all prior understandings or agreements between the parties."
3,0.945,Integration,This Agreement constitutes the entire agreement between the parties with respect to the subject matter hereof and supersedes all prior agreements.
4,0.943,Integration,This Agreement constitutes the entire agreement between the parties with respect to the subject matter hereof and supersedes all prior agreements between the parties concerning its subject matter.
5,0.941,Integration,"This Agreement constitutes the entire agreement between the parties with respect to the subject matter hereof and supersedes all prior agreements between the parties concerning such subject matter, in"


## 2. A clause written by hand, in European style

Not from LEDGAR: a governing-law and jurisdiction clause as a French company would draft it in English. LEDGAR is
built from US SEC filings, so this is outside the distribution the model was evaluated on — there is no gold label.

In [8]:
EUROPEAN_CLAUSE = (
    "This Agreement and any non-contractual obligations arising out of or in connection with it shall be governed by "
    "French law. Any dispute arising out of or in connection with this Agreement, including any question regarding "
    "its existence, validity or termination, shall be submitted to the exclusive jurisdiction of the Commercial "
    "Court of Paris (Tribunal de commerce de Paris)."
)
summary.append(show_review("Hand-written clause — European style (English)", EUROPEAN_CLAUSE))

Hand-written clause — European style (English)
This Agreement and any non-contractual obligations arising out of or in connection with it shall be governed by French law. Any dispute arising out of or in connection with this Agreement, including any question regarding its existence, validity or termination, shall be submitted to the exclusive jurisdiction of the Commercial Court of Paris (Tribunal de commerce de Paris).



,clause type,probability
1,Governing Laws,0.903
2,Applicable Laws,0.068
3,Jurisdictions,0.009


Decision: AUTOMATIC (confidence 0.903 vs threshold 0.892)


,cosine,label,first 200 characters
1,0.822,Jurisdictions,"This Agreement shall be governed by and construed in accordance with the laws of the Grand Duchy of Luxembourg. Each party agrees that all legal proceedings concerning the interpretations, enforcement"
2,0.814,Governing Laws,"All matters arising under this Agreement, including matters of validity, construction and interpretation, shall be governed by the laws of the State of Delaware, without regard to any state’s conflict"
3,0.814,Governing Laws,"This Agreement and all disputes arising out of or related to this Agreement or any breach hereof shall be governed by and construed under the laws of the State of Delaware, without giving effect to an"
4,0.812,Remedies,"This Agreement shall be governed and interpreted in accordance with the laws of the State of Delaware, and all claims relating to or arising out of this Agreement, or the breach thereof, whether sound"
5,0.810,Governing Laws,This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware without giving effect to principles of conflicts of laws. Any dispute relating to this Agreement


## 3. A French clause — the documented limitation

The model and its tokenizer are English-only (`legal-bert-base-uncased`), and the model card lists non-English
clauses as a **prohibited use**. This 30-day termination clause is shown to make the limitation visible: whatever
the output, it must not be trusted, and the confidence score is not a reliable alarm for out-of-distribution input.

In [9]:
FRENCH_CLAUSE = (
    "Le présent contrat peut être résilié par l'une ou l'autre des parties, à tout moment, moyennant un préavis écrit "
    "de trente (30) jours adressé par lettre recommandée avec accusé de réception."
)
summary.append(show_review("French clause — prohibited use (non-English)", FRENCH_CLAUSE))

French clause — prohibited use (non-English)
Le présent contrat peut être résilié par l'une ou l'autre des parties, à tout moment, moyennant un préavis écrit de trente (30) jours adressé par lettre recommandée avec accusé de réception.



,clause type,probability
1,Notices,0.214
2,Miscellaneous,0.078
3,Terms,0.061


Decision: SEND TO A LAWYER (confidence 0.214 vs threshold 0.892)


,cosine,label,first 200 characters
1,0.657,Remedies,Each Party acknowledges that the failure by the receiving Party to comply with any of the provisions of this Article 8 will result in irreparable injury and continuing damage to the disclosing Party f
2,0.638,Remedies,Each Part y agrees t h at the unauthorized u se or di sc l osure of any Co n fident i al I nformation by the R ece i ving P arty in violation o f this Agreement will cause severe an d irreparable dama
3,0.632,Remedies,"The Parties acknowledge and agree that, in the event of a breach or a threatened breach by either Party of this Agreement for which it will have no adequate remedy at law, the other Party may suffer i"
4,0.628,Severability,"The provisions of this Agreement are severable. If any provision(s) of this Agreement are adjudged void, unenforceable or contrary to law, the balance of the Agreement nonetheless will remain in full"
5,0.627,Specific Performance,"Each party hereto acknowledges that the remedies at law of the other parties for a breach or threatened breach of this Agreement would be inadequate and, in recognition of this fact, any party to this"


## 4. A lawyer's question (retrieval only)

No classification: the question is embedded and compared with the 60,000 training clauses. Same-label precision was
measured for clause queries only (notebook 05); questions like this one have no relevance judgments, so the hits
below are for reading, not a score.

In [10]:
QUESTION = "termination clauses with a 30-day notice period"
hits = pd.DataFrame(reviewer.search(QUESTION, k=5))[["score", "label_name", "snippet"]].round(3)
hits.index = range(1, len(hits) + 1)
hits.columns = ["cosine", "label", "first 200 characters"]
print(f"Question: {QUESTION}")
display(hits)

Question: termination clauses with a 30-day notice period


,cosine,label,first 200 characters
1,0.857,Terms,The term of this Agreement shall be for a period of ______ and shall be automatically extended for additional ______ terms on the same conditions as set forth herein unless earlier terminated by eithe
2,0.834,Notices,It shall be a condition precedent to XOMA’s right to terminate Employee’s employment for the reasons set forth in Sections 7(d)(ii) or (iii) of this Agreement that (i) XOMA shall first have given Empl
3,0.831,Terminations,The Company may terminate this Agreement without cause at any time upon thirty (30) days written notice to the Employee. The Employee may terminate this Agreement without cause at any time upon thirty
4,0.829,Terminations,The Company may terminate this Agreement without cause at any time upon thirty (30) days written notice to the Employee. The Employee may terminate this Agreement without cause at any time upon thirty
5,0.829,Notices,"In the event of an occurrence constituting the basis of termination for Cause, with the exception of the basis for Cause under subsections 1(a)(i), 1(a)(ii), 1(a)(iii) or 1(a)(iv) of this agreement, t"


## 5. Summary

In [11]:
display(pd.DataFrame(summary).set_index("clause"))

,gold,predicted,confidence,decision,top retrieved label
clause,,,,,
Test clause 9690,Base Salary,Base Salary,0.994,automatic,Base Salary
Test clause 9550,Waivers,Waivers,0.844,send to a lawyer,Waivers
Test clause 1201,Integration,Entire Agreements,0.960,automatic,Integration
Hand-written clause,—,Governing Laws,0.903,automatic,Jurisdictions
French clause,—,Notices,0.214,send to a lawyer,Remedies


**Reading the demo.**

- **Decision rule.** Clauses at or above 0.892 go through, the others go to a lawyer. Test clause 9550 is labelled
  correctly but sent to a lawyer (0.844: *Waivers* vs *No Waivers*, a near-synonym pair): the threshold trades coverage
  for precision. Test clause 1201 is a **confident error** that passes automatically (0.960): the threshold reduces
  errors (precision 0.949 on test), it does not remove them. Its error is the most frequent type found in notebook 03,
  a near-synonym label pair (*Integration* vs *Entire Agreements*).
- **Retrieved clauses as evidence.** For clause 1201, 4 of the 5 closest training clauses carry the gold label
  *Integration*, not the predicted one: a lawyer comparing them would look twice. This disagreement signal was not
  measured anywhere in the project, so it is a reading aid, not a validated alarm.
- **European clause.** Labelled *Governing Laws* at 0.903, just above the threshold, although it also sets the
  jurisdiction (*Jurisdictions* is third). It is outside the evaluated distribution (US SEC filings), so this output
  carries no measured accuracy.
- **French clause.** The model predicts *Notices* for a termination clause, at low confidence (0.214), and the
  retrieved clauses are unrelated (*Remedies*, *Severability*). Here the low confidence happens to send it to a lawyer,
  but one example is not evidence that non-English input is always caught: non-English clauses stay a prohibited use
  (`docs/model_card.md`).
- **Lawyer's question.** Two of the five hits (3 and 4, near-identical) are termination clauses with a thirty-day
  notice; the others are about term, notice or termination for cause. The 30-day constraint is matched by wording, not
  enforced as a constraint; numeric constraints were never evaluated (the 28 draft lawyer questions of
  `eval/retrieval_queries.jsonl` have no relevance judgments).